# CONSTRUCT-SAFE AI: Positive-Only Training Run
This notebook automates downloading the dataset, filtering out ALL negative/noisy classes (no_helmet, no_gloves, none, etc.), and training the YOLO11n custom PPE model on 6 core classes: Person, helmet, vest, boots, gloves, goggles.

In [ ]:
!pip install ultralytics roboflow
from IPython import display
display.clear_output()
import ultralytics
ultralytics.checks()

## 1. Download Dataset

In [ ]:
import os
import urllib.request
import zipfile

dataset_url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip'
zip_path = 'construction-ppe.zip'
extract_dir = '/content/datasets/construction_safety'

if not os.path.exists(extract_dir):
    urllib.request.urlretrieve(dataset_url, zip_path)
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    os.remove(zip_path)
print('Base dataset ready.')

## 2. Filter Dataset to Positive-Only
Dropping all no_* classes and keeping only: Person(0), helmet(1), vest(2), boots(3), gloves(4), goggles(5).

In [ ]:
import shutil
import glob

remap_dict = {6:0, 0:1, 2:2, 3:3, 1:4, 4:5, 5:-1, 7:-1, 8:-1, 9:-1, 10:-1}
src_dir = extract_dir
dst_dir = '/content/datasets/construction_safety_positive'

if os.path.exists(dst_dir):
    shutil.rmtree(dst_dir)

splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(f'{dst_dir}/images/{split}', exist_ok=True)
    os.makedirs(f'{dst_dir}/labels/{split}', exist_ok=True)
    lbls = glob.glob(f'{src_dir}/labels/{split}/*.txt')
    for lbl_file in lbls:
        base = os.path.basename(lbl_file)
        with open(lbl_file, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            p = line.strip().split()
            if not p: continue
            old_c = int(p[0])
            new_c = remap_dict.get(old_c, -1)
            if new_c != -1:
                new_lines.append(f"{new_c} {' '.join(p[1:])}\n")
        with open(f'{dst_dir}/labels/{split}/{base}', 'w') as f:
            f.writelines(new_lines)
        for ext in ['.jpg', '.jpeg', '.png']:
            img = f'{src_dir}/images/{split}/{os.path.splitext(base)[0]}{ext}'
            if os.path.exists(img):
                shutil.copy(img, f'{dst_dir}/images/{split}/{os.path.splitext(base)[0]}{ext}')
                break
print('Positive-only Dataset generated successfully.')

## 3. Configure Dataset YAML

In [ ]:
yaml_content = """path: /content/datasets/construction_safety_positive
train: images/train
val: images/val
test: images/test
names:
  0: Person
  1: helmet
  2: vest
  3: boots
  4: gloves
  5: goggles
"""
with open('dataset_positive.yaml', 'w') as f:
    f.write(yaml_content)

## 4. Train Model

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='dataset_positive.yaml',
    epochs=100,
    imgsz=800,
    batch=16,
    device=0,
    patience=25,
    optimizer='AdamW',
    cos_lr=True,
    mosaic=1.0,
    mixup=0.1,
    project='runs',
    name='ppe_training_positive',
    verbose=True
)

## 5. Evaluate on Test Set

In [ ]:
best_model = YOLO('runs/ppe_training_positive/weights/best.pt')
metrics = best_model.val(data='dataset_positive.yaml', split='test')
print("\nmAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)